In [ ]:
import os

def print_directory_structure(root_dir, indent=""):
    for item in os.listdir(root_dir):
        item_path = os.path.join(root_dir, item)
        if os.path.isdir(item_path):
            print(indent + "|-- " + item)
            print_directory_structure(item_path, indent + "    ")

# Set the directory path you want to print
directory_path = "/kaggle/input/nthuddd2"

print(directory_path)
print_directory_structure(directory_path)


In [1]:
import os
import random
from shutil import copy2

# Define source and destination paths
source_dir = "/kaggle/input/nthuddd2/train_data"
dest_dir = "/kaggle/working/nthuddd2_yolo"

# YOLO structure
images_train = os.path.join(dest_dir, "images", "train")
images_val = os.path.join(dest_dir, "images", "val")
labels_train = os.path.join(dest_dir, "labels", "train")
labels_val = os.path.join(dest_dir, "labels", "val")

# Create required directories
for path in [images_train, images_val, labels_train, labels_val]:
    os.makedirs(path, exist_ok=True)

# Class mapping
class_map = {"drowsy": 0, "notdrowsy": 1}

# Process images
train_ratio = 0.8  # 80% for training, 20% for validation
sample_ratio = 0.1  # Take only 10% of total dataset

for category in ["drowsy", "notdrowsy"]:
    class_id = class_map[category]
    category_path = os.path.join(source_dir, category)
    images = [f for f in os.listdir(category_path) if f.endswith(".jpg")]

    # Select 10% of the dataset randomly
    sampled_images = random.sample(images, int(len(images) * sample_ratio))
    
    random.shuffle(sampled_images)
    train_count = int(len(sampled_images) * train_ratio)

    for i, img in enumerate(sampled_images):
        img_path = os.path.join(category_path, img)
        label_content = f"{class_id} 0.5 0.5 1.0 1.0"  # Whole image as the object

        if i < train_count:
            copy2(img_path, images_train)
            label_path = os.path.join(labels_train, img.replace(".jpg", ".txt"))
        else:
            copy2(img_path, images_val)
            label_path = os.path.join(labels_val, img.replace(".jpg", ".txt"))

        with open(label_path, "w") as label_file:
            label_file.write(label_content)

print("Dataset (10% sampled) organized in YOLO format!")


Dataset organized in YOLO format!


In [3]:
dataset_yaml = """\
path: /kaggle/working/nthuddd2_yolo
train: images/train
val: images/val

names:
  0: drowsy
  1: notdrowsy
"""

# Save the YAML file
yaml_path = "/kaggle/working/dataset.yaml"
with open(yaml_path, "w") as f:
    f.write(dataset_yaml)

print(f"dataset.yaml created at {yaml_path}")


dataset.yaml created at /kaggle/working/dataset.yaml


In [5]:
!pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 922.1/922.1 kB 15.3 MB/s eta 0:00:0000:01


In [ ]:
from ultralytics import YOLO
import torch
import shutil
import os
from IPython.display import FileLink

# Check if GPU is available
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# Load YOLOv8m (pretrained for better accuracy)
model = YOLO("yolov8m.pt")

# Train the model
model.train(
    data="/kaggle/working/dataset.yaml",
    epochs=70,  # Adjusted for best accuracy
    imgsz=640,
    batch=16,
    device=device,
    workers=4,
    patience=10,
    save=True,
    save_period=10,
    optimizer="AdamW",
    cos_lr=True,
    cache=True
)

# Path to training results folder
train_folder = "/kaggle/working/runs/train/exp"

# Copy best model separately for easy access
best_model_path = "/kaggle/working/best_model.pt"
shutil.copy(f"{train_folder}/weights/best.pt", best_model_path)

# Zip the entire training folder
zip_path = "/kaggle/working/yolo_training.zip"
shutil.make_archive(zip_path.replace(".zip", ""), 'zip', train_folder)
print(f"✅ Training folder zipped successfully: {zip_path}")

# Generate clickable download links
print("✅ Training complete! Click below to download your files:")
display(FileLink(zip_path))  # Download full training folder
display(FileLink(best_model_path))  # Download best model separately


Using device: cuda
Ultralytics 8.3.81 🚀 Python-3.10.12 torch-2.5.1+cu121 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: task=detect, mode=train, model=yolov8m.pt, data=/kaggle/working/dataset.yaml, epochs=70, time=None, patience=10, batch=16, imgsz=640, save=True, save_period=10, cache=True, device=cuda, workers=4, project=None, name=train2, exist_ok=False, pretrained=True, optimizer=AdamW, verbose=True, seed=0, deterministic=True, single_cls=False, rect=False, cos_lr=True, close_mosaic=10, resume=False, amp=True, fraction=1.0, profile=False, freeze=None, multi_scale=False, overlap_mask=True, mask_ratio=4, dropout=0.0, val=True, split=val, save_json=False, save_hybrid=False, conf=None, iou=0.7, max_det=300, half=False, dnn=False, plots=True, source=None, vid_stride=1, stream_buffer=False, visualize=False, augment=False, agnostic_nms=False, classes=None, retina_masks=False, embed=None, show=False, save_frames=False, save_txt=False, save_conf=False, save_crop=False, show_labels=True, show_c

train: Scanning /kaggle/working/nthuddd2_yolo/labels/train.cache... 53216 images, 0 backgrounds, 0 corrupt: 100%|██████████| 53216/53216 [00:00<?, ?it/s]


train: 68.5GB RAM required to cache images with 50% safety margin but only 15.5/31.4GB available, not caching images ⚠️
albumentations: Blur(p=0.01, blur_limit=(3, 7)), MedianBlur(p=0.01, blur_limit=(3, 7)), ToGray(p=0.01, num_output_channels=3, method='weighted_average'), CLAHE(p=0.01, clip_limit=(1.0, 4.0), tile_grid_size=(8, 8))


val: Scanning /kaggle/working/nthuddd2_yolo/labels/val.cache... 13305 images, 0 backgrounds, 0 corrupt: 100%|██████████| 13305/13305 [00:00<?, ?it/s]


val: 17.1GB RAM required to cache images with 50% safety margin but only 14.7/31.4GB available, not caching images ⚠️
Plotting labels to runs/detect/train2/labels.jpg... 
optimizer: AdamW(lr=0.01, momentum=0.937) with parameter groups 77 weight(decay=0.0), 84 weight(decay=0.0005), 83 bias(decay=0.0)
TensorBoard: model graph visualization added ✅
Image sizes 640 train, 640 val
Using 4 dataloader workers
Logging results to runs/detect/train2
Starting training for 70 epochs...

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size


...


      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size


       2/70        12G    0.09752     0.7643     0.8803         51        640: 100%|██████████| 3326/3326 [31:28<00:00,  1.76it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 416/416 [02:36<00:00,  2.67it/s]

                   all      13305      13305      0.497      0.998      0.505      0.504



      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size


       3/70        12G    0.09026     0.7588     0.8781         51        640:  54%|█████▎    | 1780/3326 [16:49<14:28,  1.78it/s]